# TabPFN v1 — Telco Churn

TabPFN é um modelo pré-treinado de in-context learning para tabular data.
Não requer treinamento: apenas `fit` (armazena contexto) + `predict`.


In [1]:
import sys, os
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from model_utils import load_data, compute_metrics, save_results, print_scores

## 0. Instalar TabPFN

In [ ]:

from tabpfn import TabPFNClassifier
import tabpfn


## 1. Carregar dados

In [4]:
X_train, y_train, X_val, y_val, X_test, y_test = load_data()
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

Xtr_np  = X_train.values.astype('float32');  ytr_np  = y_train.values.astype(int)
Xvl_np  = X_val.values.astype('float32');    yvl_np  = y_val.values.astype(int)
Xte_np  = X_test.values.astype('float32');   yte_np  = y_test.values.astype(int)


Train: (7242, 22)  Val: (1057, 22)  Test: (1057, 22)


## 2. Subsample estratificado do treino


In [7]:
# TabPFN v1 aceita no máximo 1000 amostras de treino
MAX_TRAIN = 1000
if len(Xtr_np) > MAX_TRAIN:
    from sklearn.model_selection import train_test_split
    Xtr_sub, _, ytr_sub, _ = train_test_split(
        Xtr_np, ytr_np,
        train_size=MAX_TRAIN,
        stratify=ytr_np,
        random_state=42
    )
    print(f'Subsample: {len(Xtr_np)} → {len(Xtr_sub)} (estratificado)')
else:
    Xtr_sub, ytr_sub = Xtr_np, ytr_np
print(f'Treino completo: {len(Xtr_sub)} amostras')
print(f'Churn rate subsample: {ytr_sub.mean():.3f}')


Subsample: 7242 → 1000 (estratificado)
Treino completo: 1000 amostras
Churn rate subsample: 0.500


## 3. Fit + Predict

In [9]:
clf = TabPFNClassifier(device='cuda', N_ensemble_configurations=32)
clf.fit(Xtr_sub, ytr_sub)
print('TabPFN fit concluído')

probs = clf.predict_proba(Xte_np)[:, 1]
preds = (probs >= 0.5).astype(int)
print(f'Predictions geradas: {len(preds)} amostras de teste')


Multiple models in memory. This might lead to memory issues. Consider calling remove_models_from_memory()
TabPFN fit concluído


/home/bmmuc/Documents/repos/germano/.venv/lib/python3.11/site-packages/tabpfn/scripts/transformer_prediction_interface.py:530: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=fp16_inference):
/home/bmmuc/Documents/repos/germano/.venv/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Predictions geradas: 1057 amostras de teste


## 4. Avaliar no teste + salvar artefatos

In [10]:
scores = compute_metrics(yte_np, preds, probs)
print('Scores no teste:')
print_scores(scores)

os.makedirs('results/tabpfn', exist_ok=True)
joblib.dump(clf, 'results/tabpfn/model.pkl')

save_results('tabpfn', yte_np, preds, probs, scores)
print(f'\nNota: modelo treinado com subsample estratificado de {len(Xtr_sub)} amostras.')

Scores no teste:
  accuracy               0.7436
  balanced_accuracy      0.7399
  precision              0.5112
  recall                 0.7321
  f1                     0.6021
  auroc                  0.8282
  ks                     0.5002
Saved → results/tabpfn

Nota: modelo treinado com subsample estratificado de 1000 amostras.
